## Zadanie 2.2: 

Kiedy sieć jest bardzo głęboka, a zbiór danych niewielki, model często uczy się danych treningowych na pamięć, zamiast wyciągać ogólne reguły (tzw. przeuczenie / overfitting).

Poniżej znajduje się klasa `GlebokaSiec`. Chcemy wzbogacić ją o dwie techniki regularyzacji:
1. **Dropout:** "Wyłącza" losowe neurony podczas treningu, zmuszając pozostałe do lepszej pracy.
2. **Batch Normalization (BatchNorm):** Normalizuje aktywacje wewnątrz warstw, co przyspiesza trening i stabilizuje model.

**Twoje zadanie:**
1. W metodzie `__init__` dodaj warstwę `nn.BatchNorm2d` po pierwszej i drugiej warstwie splotowej (zastanów się, ile kanałów będzie ona normalizować).
2. W metodzie `__init__` dodaj warstwę `nn.Dropout` (np. z parametrem `p=0.5`).
3. W metodzie `forward` wpleć te warstwy w przepływ danych. Konwencja mówi, że BatchNorm stosujemy *przed* lub *po* funkcji aktywacji (często po warstwie splotowej, ale przed ReLU), a Dropout stosujemy przed warstwami liniowymi (`Linear`).

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [5]:
class GlebokaSiec(nn.Module):
    def __init__(self):
        super(GlebokaSiec, self).__init__()
        # --- BLOK 1 ---
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        # TODO: Dodaj warstwę nn.BatchNorm2d dla 32 kanałów (nazwij ją np. self.bn1)
        
        # --- BLOK 2 ---
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        # TODO: Dodaj warstwę nn.BatchNorm2d dla 64 kanałów (nazwij ją np. self.bn2)
        
        # --- BLOK 3 (Klasyfikator) ---
        # TODO: Dodaj warstwę nn.Dropout (nazwij ją np. self.dropout)
        
        self.fc1 = nn.Linear(64 * 8 * 8, 256) # Zakładamy obraz 32x32 na wejściu i 2x maxpool
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        # --- PRZEPŁYW BLOK 1 ---
        x = self.conv1(x)
        # TODO: Zastosuj self.bn1(x) tutaj
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        
        # --- PRZEPŁYW BLOK 2 ---
        x = self.conv2(x)
        # TODO: Zastosuj self.bn2(x) tutaj
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        
        # Spłaszczanie
        x = x.view(-1, 64 * 8 * 8)
        
        # --- PRZEPŁYW BLOK 3 ---
        # TODO: Zastosuj Dropout przed warstwą fc1 lub między fc1 a fc2
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        
        return x

In [8]:
# Test kompilacji
test_input = torch.randn(1, 3, 32, 32)
model_reg = GlebokaSiec()
print(model_reg(test_input).shape) # Powinno zwrócić torch.Size([1, 10])
print("Warstwy w twoim modelu:")
print(model_reg) # Wydrukuj model, aby sprawdzić, czy Dropout i BatchNorm są na liście

torch.Size([1, 10])
Warstwy w twoim modelu:
GlebokaSiec(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=4096, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=10, bias=True)
)
